In [24]:
import json
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import os
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import (
    FaithfulnessMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    AnswerRelevancyMetric
)
from deepeval.evaluate import AsyncConfig
from deepeval.models import OllamaModel

# 📊 Définition des Métriques d'Évaluation RAG

## 🎯 Métriques utilisées

Ce benchmark évalue un système RAG avec 4 métriques DeepEval :

### 1️⃣ **Faithfulness (Fidélité)** - Score: 0 à 1

**Mesure** : La réponse générée est-elle fidèle au contexte récupéré, sans hallucinations?

- Extrait les vérités du contexte et les affirmations de la réponse
- Vérifie que chaque affirmation peut être justifiée par le contexte
- Détecte les informations inventées ou inexactes

**Interprétation** :
- 0.8+ : Excellente fidélité
- 0.5-0.8 : Bonne fidélité avec quelques imprécisions
- <0.5 : Hallucinations détectées

---

### 2️⃣ **Contextual Precision (Précision Contextuelle)** - Score: 0 à 1

**Mesure** : Quel pourcentage des documents récupérés sont vraiment pertinents?

- Évalue si chaque document retourné contient des informations utiles
- Calcule: documents pertinents / documents totaux récupérés
- Ignore l'ordre de classement

**Interprétation** :
- 1.0 : Tous les documents sont pertinents
- 0.7-0.9 : Bonne précision, peu de bruit
- 0.5-0.7 : Acceptable, du bruit présent
- <0.5 : Mauvaise précision

---

### 3️⃣ **Contextual Recall (Rappel Contextuel)** - Score: 0 à 1

**Mesure** : TOUTES les informations nécessaires sont-elles présentes dans les documents récupérés?

- Extrait les phrases clés de la réponse attendue
- Vérifie si chaque phrase clé existe dans les documents récupérés
- Calcule: phrases trouvées / phrases totales

**Interprétation** :
- 1.0 : Toutes les infos nécessaires sont présentes
- 0.7-0.9 : La plupart des infos sont présentes
- 0.5-0.7 : Environ la moitié des infos manquent
- <0.5 : Trop d'infos manquantes

---

### 4️⃣ **Answer Relevancy (Pertinence de la Réponse)** - Score: 0 à 1

**Mesure** : La réponse générée répond-elle vraiment à la question posée?

- Analyse la cohérence sémantique entre la question et la réponse
- Détecte les informations hors-sujet
- Calcule: infos pertinentes / infos totales

**Interprétation** :
- 0.8+ : Très pertinent
- 0.6-0.8 : Pertinent avec quelques hors-sujets
- 0.4-0.6 : Partiellement pertinent
- <0.4 : Peu pertinent

---

## 📈 Score Global

```
Score moyen = (Faithfulness + Precision + Recall + Relevancy) / 4

Excellent:  0.8-1.0 ✅✅✅
Bon:        0.6-0.8 ✅✅
Acceptable: 0.4-0.6 ✅
Faible:     <0.4    ❌
```

**Pass rate** = Pourcentage de métriques qui dépassent le seuil de 0.5

In [25]:
# -------------------------------------------------
# ⚙️ PARAMÈTRES DU BENCHMARK
# -------------------------------------------------
TOP_K = 5        # nombre de documents à récupérer

# Chargement des modèles
retriever_model = SentenceTransformer("BAAI/bge-m3")

In [26]:
# -------------------------------------------------
# 🔹 CHARGEMENT DES DONNÉES
# -------------------------------------------------
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

dataset = load_json("./test_data.json")  # contient queries, corpus, ground_truth
queries = dataset["queries"]
corpus = dataset["corpus"]
ground_truth = dataset["ground_truth"]

corpus_ids = [d["id"] for d in corpus]
corpus_texts = [d["text"] for d in corpus]

In [27]:
# -------------------------------------------------
# 🔹 ÉTAPE 1 — RETRIEVAL
# -------------------------------------------------
def get_embeddings(model, texts):
    return model.encode(texts, normalize_embeddings=True)

print("\n🔹 Génération des embeddings du corpus...")
corpus_emb = get_embeddings(retriever_model, corpus_texts)

def retrieve(query, top_k=TOP_K):
    q_emb = get_embeddings(retriever_model, [query])[0]
    sims = cosine_similarity([q_emb], corpus_emb)[0]
    top_idx = np.argsort(-sims)[:top_k]
    return [corpus[idx] for idx in top_idx]


🔹 Génération des embeddings du corpus...


In [31]:
# -------------------------------------------------
# 🔹 ÉTAPE 2 — GÉNÉRATION DE LA RÉPONSE
# -------------------------------------------------
import sys
sys.path.append('/Users/sedki/Desktop/Projet INEO/llm_rag_2')

from search_and_rerank import search_and_rerank

def generate_answer(question):
    query = question
    results = search_and_rerank(query)
    
    if not results:
        print("❌ Aucun résultat trouvé.")
        return "", []
    
    print(f"\n🔍 Résultats pour : {query}\n")
    response_final = ""
    chunks_list = []  # ✅ Liste pour retrieval_context
    
    for r in results:
        phrase = f"{r['rank']}.  {r['doc']} (Page {r['page']})\n"
        phrase += f"    {r['chunk'][:200]}...\n"
        phrase += f"    Pertinence : {r['rerank_score']:.4f} |  Score Qdrant : {r['qdrant_score']:.4f}\n"
        phrase += "-" * 100 + "\n"
        response_final += phrase
        chunks_list.append(r['chunk'])  # ✅ Ajoute le chunk à la liste

    return response_final.strip(), chunks_list

In [36]:
# -------------------------------------------------
# 🔹 ÉTAPE 3 — ÉVALUATION GÉNÉRATION (DeepEval + Ollama)
# -------------------------------------------------

# Configuration Ollama
eval_llm = OllamaModel(model="Neural-Chat")

def evaluate_generation(queries, ground_truth):
    print("\n🔹 Évaluation avec DeepEval + Ollama...")
    print(f"📌 Total de questions à évaluer: {len(queries)}\n")

    test_cases = []
    
    # 🔹 Évalue TOUTES les questions
    for idx, q in enumerate(queries[:3]):
        print(f"⏳ Traitement question {idx + 1}/{len(queries)}: {q['text'][:80]}...")
        
        qid = q["id"]
        relevant_docs = [c for c in corpus if c["id"] in ground_truth.get(qid, [])]
        retrieved_docs = retrieve(q["text"])

        # Génération
        response, chunks_list = generate_answer(q["text"])  
        expected_answer = "\n".join([d["text"] for d in relevant_docs])

        # Crée LLMTestCase
        test_case = LLMTestCase(
            input=q["text"],
            actual_output=response,
            expected_output=expected_answer,
            retrieval_context=chunks_list  
        )
        test_cases.append(test_case)

    # Utilise Ollama pour les métriques
    metrics = [
        FaithfulnessMetric(model=eval_llm),
        ContextualPrecisionMetric(model=eval_llm),
        ContextualRecallMetric(model=eval_llm),
        AnswerRelevancyMetric(model=eval_llm),
    ]

    print(f"\n🔍 Évaluation des {len(test_cases)} test cases...\n")
    results = evaluate(
        test_cases, 
        metrics=metrics,
        async_config=AsyncConfig(run_async=False)
    )
    
    print("\n" + "="*80)
    print("📊 RÉSULTATS FINAUX")
    print("="*80)
    print(results)
    print("="*80)

In [37]:
# -------------------------------------------------
# 🚀 MAIN
# -------------------------------------------------
if __name__ == "__main__":
    evaluate_generation(queries, ground_truth)



🔹 Évaluation avec DeepEval + Ollama...
📌 Total de questions à évaluer: 40

⏳ Traitement question 1/40: Comment la transformée de Fourier rapide (FFT) améliore-t-elle l'efficacité du c...

🔍 Résultats pour : Comment la transformée de Fourier rapide (FFT) améliore-t-elle l'efficacité du calcul spectral par rapport à la transformée de Fourier discrète standard ?

⏳ Traitement question 2/40: Quelles sont les méthodes les plus efficaces pour réduire le bruit blanc gaussie...

🔍 Résultats pour : Quelles sont les méthodes les plus efficaces pour réduire le bruit blanc gaussien dans les signaux EEG tout en préservant les composantes neurologiques importantes ?

⏳ Traitement question 3/40: Comment les coefficients cepstraux MFCC sont-ils calculés et pourquoi sont-ils s...

🔍 Résultats pour : Comment les coefficients cepstraux MFCC sont-ils calculés et pourquoi sont-ils si efficaces pour la reconnaissance automatique de la parole ?


🔍 Évaluation des 3 test cases...



✨ You're running DeepEval's latest Faithfulness Metric! (using Neural-Chat (Ollama), strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using Neural-Chat (Ollama), strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using Neural-Chat (Ollama), strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using Neural-Chat (Ollama), strict=False, 
async_mode=False)...

/Users/sedki/Desktop/Projet INEO/llm_rag_2/.venv/lib/python3.13/site-packages/rich/live.py:256: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Metrics Summary

  - ✅ Faithfulness (score: 1.0, threshold: 0.5, strict: False, evaluation model: Neural-Chat (Ollama), reason: With a faithfulness score of 1.00, there are no contradictions between the actual output and the retrieval context, indicating perfect alignment., error: None)
  - ✅ Contextual Precision (score: 1.0, threshold: 0.5, strict: False, evaluation model: Neural-Chat (Ollama), reason: The contextual precision score is 1.00 because the first relevant node (rank 1) directly discusses FFT efficiency compared to standard DFT, while irrelevant nodes (ranks 2+) focus on related but not directly connected concepts., error: None)
  - ✅ Contextual Recall (score: 1.0, threshold: 0.5, strict: False, evaluation model: Neural-Chat (Ollama), reason: Le score est 1 parce que tous les éléments clés du contexte de rétréviation sont mentionnés dans l'extrait, offrant une bonne correspondance et une compréhension globale des concepts., error: None)
  - ✅ Answer Relevancy (score: 0.57

⚠ WARNING: No hyperparameters logged.
» ]8;id=798134;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 339.2s | token cost: 0.0 USD)
» Test Results (3 total tests):
   » Pass Rate: 33.33% | Passed: 1 | Failed: 2

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.


📊 RÉSULTATS FINAUX
test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Faithfulness', threshold=0.5, success=True, score=1.0, reason='With a faithfulness score of 1.00, there are no contradictions between the actual output and the retrieval context, indicating perfect alignment.', strict_mode=False, evaluation_model='Neural-Chat (Ollama)', error=None, evaluation_cost=0.0, verbose_logs='Truths (limit=None):\n[\n    "There exists a Fourier transform called Fast Fourier Transform (FFT).",\n    "The Fourier transform is used in various applications, including Discrete Wavelet Transform and Time-Frequency Analysis."\n] \n \nClaims:\n[\n    "These documents discuss various signal processing techniques, including Fourier transform and wavelet analysis.",\n    "Fourier transform is mentioned in multiple papers related to signal processing."\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": null\n    },\n    {\n        "verdict": "y